In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from src.preprocess import BasicDegenderizer, AdvancedDegenderizer
from src.models import DistilBERTClassifier

/usr/local/lib/python3.10/dist-packages/torchvision/datapoints/__init__.py:14: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/usr/local/lib/python3.10/dist-packages/torchvision/transforms/v2/__init__.py:64: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https:/

In [2]:
DATA_PATH = "data/sentence_sets_trimmed.csv"
LABEL_COLUMN = "applicant_gender"
TEXT_COLUMN = "s1_s2"

DEGENDERIZERS = [
    "src/degenderizer-mappings/nouns.txt",
    "src/degenderizer-mappings/pronouns.txt",
    "src/degenderizer-mappings/titles.txt",
]

In [3]:
# Load dataset
df = pd.read_csv(DATA_PATH, encoding="ISO-8859-1")
print("Dataset shape:", df.shape)

Dataset shape: (3285, 19)


In [4]:
# Create degendering pipelines for each degenderizer
basic_pipeline = Pipeline([("basic", BasicDegenderizer(paths=DEGENDERIZERS))])
advanced_pipeline = Pipeline([("advanced", AdvancedDegenderizer(paths=DEGENDERIZERS))])

In [5]:
# Apply both pipelines to the text column
df["basic_degendered"] = basic_pipeline.transform(df[TEXT_COLUMN].tolist())
df["advanced_degendered"] = advanced_pipeline.transform(df[TEXT_COLUMN].tolist())

print(df[[TEXT_COLUMN, "basic_degendered", "advanced_degendered"]].head())

                                               s1_s2  \
0  FIRST_NAME LAST_NAME  * LAST_NAME waived his r...   
1  this letter is written in support of FIRST_NAM...   
2  comments from anesthesiology rotations    FIRS...   
3  long number ms LAST_NAME had originally planne...   
4  i think [ FIRST_NAME ] would be an asset to an...   

                                    basic_degendered  \
0  FIRST_NAME LAST_NAME  * LAST_NAME waived his r...   
1  this letter is written in support of FIRST_NAM...   
2  comments from anesthesiology rotations    FIRS...   
3  long number ms LAST_NAME had originally planne...   
4  i think [ FIRST_NAME ] would be an asset to an...   

                                 advanced_degendered  
0  FIRST_NAME LAST_NAME  * LAST_NAME waived his r...  
1  this letter is written in support of FIRST_NAM...  
2  comments from anesthesiology rotations    FIRS...  
3  long number ms LAST_NAME had originally planne...  
4  i think [ FIRST_NAME ] would be an asset to an..

In [6]:
# For training, we use advanced
df["degendered_text"] = df["advanced_degendered"]

In [7]:
# Convert categorical labels to factors / integers
df[LABEL_COLUMN], class_mapping = pd.factorize(df[LABEL_COLUMN])
print("Class mapping:", dict(enumerate(class_mapping)))

Class mapping: {0: 'male', 1: 'female'}


In [8]:
# Train test splits
X_train, X_test, y_train, y_test = train_test_split(
    df["degendered_text"],
    df[LABEL_COLUMN],
    test_size=0.2,
    random_state=42,
    stratify=df[LABEL_COLUMN],
)

print("Train size:", len(X_train), "Test size:", len(X_test))

Train size: 2628 Test size: 657


In [9]:
# DistilBERT classifier model
model = DistilBERTClassifier(
    model_name="distilbert-base-uncased",
    num_labels=len(class_mapping),
)

In [10]:
model.train(
    X_train.tolist(),
    y_train.tolist(),
    epochs=3,
    batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    output_dir="../scratch/models/distilbert_degendered"
)

Map:   0%|          | 0/2628 [00:00<?, ? examples/s]

/home/hice1/mwesley32/.local/lib/python3.10/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/home/hice1/mwesley32/NLP-Letters-V2/src/models.py:159: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.002480,1.000000,1.000000,1.000000,1.000000
2,No log,0.001107,1.000000,1.000000,1.000000,1.000000
3,No log,0.000873,1.000000,1.000000,1.000000,1.000000


{'eval_loss': 0.0024803257547318935,
 'eval_accuracy': 1.0,
 'eval_precision': 1.0,
 'eval_recall': 1.0,
 'eval_f1': 1.0,
 'eval_runtime': 3.1156,
 'eval_samples_per_second': 168.827,
 'eval_steps_per_second': 10.592,
 'epoch': 3.0}

In [12]:
metrics = model.test(X_test.tolist()[0:5], y_test.tolist()[0:5])

print("Evaluation Metrics:", metrics)

Evaluation Metrics: {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}


In [16]:
basic_pipeline


Pipeline(steps=[('basic',
                 BasicDegenderizer(paths=['src/degenderizer-mappings/nouns.txt',
                                          'src/degenderizer-mappings/pronouns.txt',
                                          'src/degenderizer-mappings/titles.txt']))])

In [19]:
df["advanced_degendered"] = advanced_pipeline.fit_transform(df[TEXT_COLUMN].tolist())


In [25]:
df["advanced_degendered"][0:5].tolist()

["FIRST_NAME LAST_NAME  * LAST_NAME waived their right to review this letter  * LAST_NAME received a clinical score of outstanding   a theylf board score of satisfactory   and an overall core * LAST_NAME received these comments on their core * \x94 attending e FIRST_NAME performed well on the rotation   specific strengths included actively seeking opportunities for patient care   using strong communication strategies to engender trust with patients   asking for continuous feedback and seeking opportunities to educate team   opportunities infections in hiv    *   attending e it was a pleasure working with FIRST_NAME on wards  * FIRST_NAME's a team player   gets along well with other staff and patients and family  * FIRST_NAME theylped with team work and was active member of the team  * FIRST_NAME has confidence in themself and enjoys the profession they is about to practice   they has a great attitude   that makes working with them to be fun  * \x94 resident in summary   FIRST_NAME comp

In [26]:
model.test(df["advanced_degendered"][0:5].tolist(), df[LABEL_COLUMN][0:5])

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'accuracy': 0.6, 'precision': 0.3, 'recall': 0.5, 'f1': 0.375}

In [ ]:
# Apply both pipelines to the text column
df["basic_degendered"] = basic_pipeline.transform(df[TEXT_COLUMN].tolist())
df["advanced_degendered"] = advanced_pipeline.transform(df[TEXT_COLUMN].tolist())

print(df[[TEXT_COLUMN, "basic_degendered", "advanced_degendered"]].head())

In [15]:
df["s1_s2"][0]

"FIRST_NAME LAST_NAME  * LAST_NAME waived his right to review this letter  * LAST_NAME received a clinical score of outstanding   a shelf board score of satisfactory   and an overall core * LAST_NAME received these comments on his core * \x94 attending e FIRST_NAME performed well on the rotation   specific strengths included actively seeking opportunities for patient care   using strong communication strategies to engender trust with patients   asking for continuous feedback and seeking opportunities to educate team   opportunities infections in hiv    *   attending e it was a pleasure working with FIRST_NAME on wards  * FIRST_NAME's a team player   gets along well with other staff and patients and family  * FIRST_NAME helped with team work and was active member of the team  * FIRST_NAME has confidence in himself and enjoys the profession he is about to practice   he has a great attitude   that makes working with him to be fun  * \x94 resident in summary   FIRST_NAME completed internal

In [21]:
df["advanced_degendered"][0:5]

"FIRST_NAME LAST_NAME  * LAST_NAME waived their right to review this letter  * LAST_NAME received a clinical score of outstanding   a theylf board score of satisfactory   and an overall core * LAST_NAME received these comments on their core * \x94 attending e FIRST_NAME performed well on the rotation   specific strengths included actively seeking opportunities for patient care   using strong communication strategies to engender trust with patients   asking for continuous feedback and seeking opportunities to educate team   opportunities infections in hiv    *   attending e it was a pleasure working with FIRST_NAME on wards  * FIRST_NAME's a team player   gets along well with other staff and patients and family  * FIRST_NAME theylped with team work and was active member of the team  * FIRST_NAME has confidence in themself and enjoys the profession they is about to practice   they has a great attitude   that makes working with them to be fun  * \x94 resident in summary   FIRST_NAME compl

In [20]:
import torch
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Cached: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

Allocated: 11.82 GB
Cached: 11.86 GB


In [17]:
torch.cuda.empty_cache()

In [16]:
import gc
gc.collect()

3138